# ResepAI — Google Colab Ingestion Notebook

This notebook:
1. Downloads the Indonesian Recipes dataset from HuggingFace
2. Embeds all 66K recipes using GPU-accelerated `all-MiniLM-L6-v2`
3. Stores embeddings in ChromaDB
4. Exports the ChromaDB data for deployment

**Runtime → Change runtime type → T4 GPU**

## Step 1: Setup — Install Dependencies

In [ ]:
!pip install -q chromadb==0.5.23 transformers==4.46.3 torch==2.5.1 huggingface_hub==0.26.5 pyarrow==18.1.0 pandas==2.2.3 tqdm==4.67.1

## Step 2: Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

## Step 3: Set Your HF Token

Get your token at https://huggingface.co/settings/tokens
The dataset `junwatu/indonesian-recipes` is gated — you must accept terms on the dataset page first.

In [ ]:
import os

# 👇 Replace with your actual HF token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token set:", "✓" if HF_TOKEN != "YOUR_HF_TOKEN_HERE" else "❌ Please set your token!")

## Step 4: Download Dataset from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
import pyarrow.parquet as pq
import json
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)
JSON_PATH = DATA_DIR / "recipes.jsonl"

if JSON_PATH.exists():
    print("Dataset JSON already exists, skipping download.")
    with open(JSON_PATH) as f:
        total = sum(1 for _ in f)
    print(f"Found {total} recipes in local JSON.")
else:
    print("Downloading dataset from HuggingFace...")
    parquet_path = hf_hub_download(
        repo_id="junwatu/indonesian-recipes",
        filename="data/train.parquet",
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=str(DATA_DIR),
        local_dir_use_symlinks=False,
    )
    print(f"Downloaded parquet to: {parquet_path}")

    # Convert parquet → JSON lines
    print("Converting parquet to JSON lines...")
    table = pq.read_table(parquet_path)
    df = table.to_pandas()
    
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            f.write(json.dumps(row.to_dict(), ensure_ascii=False) + "\n")
    
    print(f"✅ Converted {len(df)} recipes to {JSON_PATH}")
    print(f"Columns: {list(df.columns)}")
    print(f"\nSample recipe:")
    sample = df.iloc[0]
    print(f"  Title: {sample.get('title', 'N/A')}")
    print(f"  Ingredients: {str(sample.get('ingredients', []))[:100]}...")
    print(f"  Steps: {str(sample.get('steps', []))[:100]}...")

## Step 5: Load Embedding Model (GPU)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 256  # GPU can handle large batches

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).cuda().eval()
print("✅ Model loaded on GPU.")

@torch.no_grad()
def embed_batch(texts):
    """Embed a batch of texts, return normalized vectors."""
    encoded = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to("cuda")
    outputs = model(**encoded)
    # Mean pooling with attention mask
    mask = encoded["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
    pooled = torch.sum(outputs.last_hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
    return F.normalize(pooled, p=2, dim=1).cpu().numpy()

print("✅ Embedding function ready.")

## Step 6: Start ChromaDB + Ingest All Recipes

In [ ]:
import chromadb
from chromadb.config import Settings
from tqdm import tqdm
import json
import numpy as np
import time

CHROMA_DIR = "/content/chroma_data"
COLLECTION_NAME = "recipes"

# Start ChromaDB (persistent, in-memory server mode via PersistentClient)
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if re-running
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Deleted existing collection.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ ChromaDB collection '{COLLECTION_NAME}' created.")

# Load recipes
print("Loading recipes from JSON...")
recipes = []
with open(JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                recipes.append(json.loads(line))
            except json.JSONDecodeError:
                pass

print(f"Loaded {len(recipes)} recipes.")

# Build documents
def build_document(recipe, idx):
    title = str(recipe.get("title", recipe.get("name", f"Recipe {idx+1}"))).strip()
    
    ingredients = recipe.get("ingredients", recipe.get("ingredient", recipe.get("bahan", [])))
    if isinstance(ingredients, str):
        ingredients = [i.strip() for i in ingredients.replace(";", "\n").split("\n") if i.strip()]
    ingredients = [str(i).strip() for i in ingredients if i]
    
    steps = recipe.get("steps", recipe.get("step", recipe.get("instructions", recipe.get("langkah", []))))
    if isinstance(steps, str):
        steps = [s.strip() for s in steps.replace(";", "\n").split("\n") if s.strip()]
    steps = [str(s).strip() for s in steps if s]
    
    doc = f"Title: {title}\nIngredients: {', '.join(ingredients) or 'N/A'}\nSteps: {' | '.join(steps) or 'N/A'}"
    
    return {
        "id": f"recipe-{recipe.get('id', recipe.get('_id', idx))}",
        "document": doc,
        "metadata": {
            "title": title,
            "num_ingredients": len(ingredients),
            "num_steps": len(steps),
            "source": "indonesian-recipes",
        }
    }

print("Building documents...")
docs = [build_document(r, i) for i, r in enumerate(recipes)]
print(f"Built {len(docs)} documents.")

# Embed and ingest in batches
print(f"\n🚀 Starting ingestion (batch_size={BATCH_SIZE}, GPU-accelerated)...")
start_time = time.time()
total_ingested = 0

for i in tqdm(range(0, len(docs), BATCH_SIZE), desc="Ingesting"):
    batch = docs[i:i+BATCH_SIZE]
    ids = [d["id"] for d in batch]
    documents = [d["document"] for d in batch]
    metadatas = [d["metadata"] for d in batch]
    
    embeddings = embed_batch(documents).tolist()
    
    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings,
    )
    total_ingested += len(batch)

elapsed = time.time() - start_time
print(f"\n✅ Ingestion complete!")
print(f"   Total recipes: {total_ingested}")
print(f"   Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {total_ingested/elapsed:.0f} recipes/sec")

# Verify
count = collection.count()
print(f"\n📊 Collection count: {count}")

## Step 7: Quick Test Query

In [ ]:
# Test: search for a recipe
test_query = "nasi goreng"
query_emb = embed_batch([test_query]).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Query: '{test_query}'\n")
for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    similarity = 1 - dist
    print(f"--- Result {i+1} (similarity: {similarity:.2f}) ---")
    print(f"Title: {meta['title']}")
    print(f"Ingredients: {meta['num_ingredients']} | Steps: {meta['num_steps']}")
    print(f"Doc preview: {doc[:150]}...")
    print()

## Step 8: Export ChromaDB Data

This creates a zip file you can download and deploy.

In [ ]:
import shutil
from google.colab import files

# Zip the ChromaDB data
ZIP_PATH = "/content/chroma_export.zip"
shutil.make_archive("/content/chroma_export", "zip", CHROMA_DIR)

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"✅ Exported ChromaDB data: {ZIP_PATH}")
print(f"   Size: {size_mb:.1f} MB")
print(f"\n⬇️ Downloading...")

files.download(ZIP_PATH)
print("\n✅ Download complete!")
print("\n📋 Next steps:")
print("1. Unzip chroma_export.zip on your server")
print("2. Point your backend ChromaDB to this data directory")
print("3. Start the backend — it will use the pre-embedded data")

## Step 9 (Alternative): Export as JSON for Any Deployment

If you prefer to import the data into a fresh ChromaDB instance on your server:

In [ ]:
# Export all data from ChromaDB as JSON
all_data = collection.get(include=["documents", "metadatas", "embeddings"])

export = {
    "ids": all_data["ids"],
    "documents": all_data["documents"],
    "metadatas": all_data["metadatas"],
    "embeddings": [emb.tolist() if isinstance(emb, np.ndarray) else emb for emb in all_data["embeddings"]],
}

EXPORT_PATH = "/content/chroma_export.json"
with open(EXPORT_PATH, "w") as f:
    json.dump(export, f)

size_mb = os.path.getsize(EXPORT_PATH) / 1e6
print(f"✅ Exported as JSON: {EXPORT_PATH} ({size_mb:.1f} MB)")
print("\nOn your server, run the import script below to load this into ChromaDB.")

files.download(EXPORT_PATH)